# The Dirty Ledger — Data Cleaning

This notebook implements the cleaning and checking steps discussed:

1. Load and inspect the original CSV.
2. Identify exact duplicate rows and remove only exact duplicates.
3. Standardise item-name capitalisation (for example, `oil b` → `Oil B`).
4. Flag missing, non-positive, and statistically unusual prices.
5. Export the cleaned data to Excel with columns sized to their contents.

**Important:** Keep `teaching_prices_messy.csv` in the same folder as this notebook. The notebook does not overwrite the original CSV. Suspicious prices are flagged for review, not automatically changed or deleted.

In [ ]:
import pandas as pd
import numpy as np

# Load the original dataset
df = pd.read_csv("teaching_prices_messy.csv")

# Keep a copy of the raw data for comparison
df_original = df.copy()

print("Dataset shape:", df.shape)
display(df.head())
print("\nColumn information:")
df.info()


## 1. Initial data-quality checks

Review missing values, exact duplicate rows, and distinct values before making changes.

In [ ]:
print("Missing values by column:")
display(df.isna().sum().to_frame("missing_count"))

print("Exact duplicate rows:", df.duplicated().sum())

for column in df.select_dtypes(include="object").columns:
    print(f"\nDistinct values in {column}:")
    print(df[column].value_counts(dropna=False))


## 2. Remove exact duplicate rows

This removes only rows where **every original column** matches. Repeated item or city names are preserved because they may represent different observations.

In [ ]:
duplicates_before = df.duplicated().sum()
print("Exact duplicate rows before removal:", duplicates_before)

df = df.drop_duplicates(keep="first").copy()

duplicates_after = df.duplicated().sum()
print("Exact duplicate rows after removal:", duplicates_after)
print("Rows removed:", duplicates_before - duplicates_after)


## 3. Standardise item names

Trim leading/trailing spaces and apply title case. This standardises variants such as `oil b`, `OIL B`, and `Oil B` to `Oil B`.

This assumes differences in capitalisation do not represent different products.

In [ ]:
if "item" in df.columns:
    df["item"] = df["item"].astype("string").str.strip().str.title()

    print("Standardised item values:")
    print(sorted(df["item"].dropna().unique()))
else:
    raise KeyError("Expected an 'item' column was not found.")


## 4. Flag missing, invalid, and unusual prices

- Missing or unparseable prices are flagged.
- Zero or negative prices are flagged for review (rather than automatically removed).
- The IQR rule identifies unusually low/high prices **within the same item and city**.
- Flags are review prompts, not proof that a value is wrong.

The original `price` column is preserved. A separate numeric helper column is created for analysis.

In [ ]:
if "price" not in df.columns:
    raise KeyError("Expected a 'price' column was not found.")

# Convert price text to numeric for analysis; remove comma separators such as "1,250"
df["price_numeric"] = pd.to_numeric(
    df["price"].astype("string").str.replace(",", "", regex=False),
    errors="coerce"
)

# Identify missing or unparseable prices
df["missing_price"] = df["price_numeric"].isna()

# Flag zero or negative prices for investigation
df["nonpositive_price"] = df["price_numeric"].le(0).fillna(False)

# Calculate IQR limits within each item/city group
group_cols = ["item", "city"]
if not all(col in df.columns for col in group_cols):
    raise KeyError("The dataset must contain 'item' and 'city' columns.")

grouped_prices = df.groupby(group_cols)["price_numeric"]
q1 = grouped_prices.transform(lambda values: values.quantile(0.25))
q3 = grouped_prices.transform(lambda values: values.quantile(0.75))
iqr = q3 - q1

lower_limit = q1 - 1.5 * iqr
upper_limit = q3 + 1.5 * iqr

df["unusual_price"] = (
    df["price_numeric"].lt(lower_limit) |
    df["price_numeric"].gt(upper_limit)
).fillna(False)

# Give each row one main review status
df["price_flag"] = np.select(
    [
        df["missing_price"],
        df["nonpositive_price"],
        df["unusual_price"]
    ],
    [
        "MISSING / UNREADABLE PRICE",
        "ZERO OR NEGATIVE PRICE — REVIEW",
        "UNUSUAL PRICE — REVIEW"
    ],
    default="OK"
)

review_columns = [
    col for col in ["date", "item", "city", "price", "price_flag"]
    if col in df.columns
]

print("Records requiring price review:")
display(df.loc[df["price_flag"] != "OK", review_columns])


## 5. Before-and-after summary

Compare the original dataset with the cleaned dataset. Missing values and unusual prices are retained and flagged; they are not silently deleted.

In [ ]:
summary = pd.DataFrame({
    "Before": [
        len(df_original),
        int(df_original.isna().sum().sum()),
        int(df_original.duplicated().sum())
    ],
    "After": [
        len(df),
        int(df.isna().sum().sum()),
        int(df.duplicated().sum())
    ]
}, index=["Rows", "Missing cells (all columns)", "Exact duplicate rows"])

display(summary)

print("Price review flags:")
display(df["price_flag"].value_counts(dropna=False).to_frame("count"))


## 6. Export to Excel and automatically fit column widths

This is a presentation adjustment only; it does not change the underlying dates or other data. The export requires `openpyxl`.

In [ ]:
# Export the cleaned dataset
output_file = "cleaned_prices.xlsx"
df.to_excel(output_file, index=False)

# Adjust Excel column widths to fit their contents
from openpyxl import load_workbook
from openpyxl.utils import get_column_letter

workbook = load_workbook(output_file)
worksheet = workbook.active

for column_cells in worksheet.columns:
    max_length = 0
    column_index = column_cells[0].column
    column_letter = get_column_letter(column_index)

    for cell in column_cells:
        if cell.value is not None:
            max_length = max(max_length, len(str(cell.value)))

    worksheet.column_dimensions[column_letter].width = min(max_length + 2, 50)

workbook.save(output_file)
print(f"Saved: {output_file}")


## 7. Final verification and notes

Before submitting, review the flagged records manually. In particular:

- Confirm whether each exact duplicate was genuinely redundant.
- Check whether unusual prices (such as a very high value) are data-entry errors or legitimate observations.
- Do not fill in missing prices without a defensible source or documented method.
- Confirm that standardising item names has not merged genuinely different products.

**Remember:** An outlier is not automatically an error. Document the evidence and your decision for each flagged record.